# 02 · `match` / `case`: Structural Pattern Matching 🆕 (Python 3.10+)

**Module 04 – Control Flow** · Estimated time: 50 min

> 📘 The handbook introduces `match-case` with one small example. 📗 The bootcamp (written before 3.10) doesn't cover it at all. This lesson covers it in full.

### 🎯 What you will learn
- Basic `match` on literal values, with a default `case _`
- OR-patterns `|`, capture patterns, guards `if`
- Sequence patterns (lists/tuples), with `*rest`
- Mapping patterns (dicts), e.g. for JSON data
- Class patterns (match on object types and attributes)
- `as` bindings
- ❌ The #1 pitfall: bare names capture, they don't compare

## 1. Basic syntax

```python
match subject:
    case pattern1:
        ...
    case pattern2:
        ...
    case _:          # wildcard: matches anything (the "default")
        ...
```
Patterns are tried top to bottom; the **first** that matches wins. There is no fall-through.

In [1]:
def http_status(status):
    match status:
        case 200:
            return "Success!"
        case 404:
            return "Not Found"
        case 500:
            return "Server Error"
        case _:
            return "Unknown Status"

for code in [200, 404, 500, 418]:
    print(code, http_status(code))

200 Success!
404 Not Found
500 Server Error
418 Unknown Status


Without a `case _`, if nothing matches, nothing happens (no error).

## 2. OR-patterns with `|`

In [2]:
def day_type(day):
    match day.lower():
        case "saturday" | "sunday":
            return "Weekend 🎉"
        case "monday" | "tuesday" | "wednesday" | "thursday" | "friday":
            return "Weekday"
        case _:
            return "Not a day"

print(day_type("Sunday"), day_type("Wednesday"), day_type("Funday"))

Weekend 🎉 Weekday Not a day


## 3. Capture patterns & guards

A **bare name** in a pattern *captures* the value into that variable. A **guard** (`if ...`) adds an extra condition.

In [3]:
def describe(n):
    match n:
        case 0:
            return "zero"
        case x if x < 0:
            return f"negative ({x})"
        case x if x % 2 == 0:
            return f"positive even ({x})"
        case x:
            return f"positive odd ({x})"

print([describe(v) for v in [0, -3, 4, 7]])

['zero', 'negative (-3)', 'positive even (4)', 'positive odd (7)']


## 4. Sequence patterns (lists & tuples)

Match on **shape** and pull values out at the same time.

In [4]:
def handle(command):
    match command.split():
        case ["quit"] | ["exit"]:
            return "Goodbye!"
        case ["go", direction]:
            return f"Going {direction}"
        case ["get", item]:
            return f"Picked up {item}"
        case ["drop", *items]:                 # *rest captures the remainder
            return f"Dropped {', '.join(items)}"
        case []:
            return "Say something!"
        case _:
            return f"Unknown command: {command!r}"

for c in ["go north", "get lamp", "drop sword shield key", "quit", "", "dance wildly"]:
    print(f"{c!r:28} → {handle(c)}")

'go north'                   → Going north
'get lamp'                   → Picked up lamp
'drop sword shield key'      → Dropped sword, shield, key
'quit'                       → Goodbye!
''                           → Say something!
'dance wildly'               → Unknown command: 'dance wildly'


In [5]:
def where(point):
    match point:
        case (0, 0):
            return "Origin"
        case (0, y):
            return f"On the Y axis at y={y}"
        case (x, 0):
            return f"On the X axis at x={x}"
        case (x, y):
            return f"At ({x}, {y})"
        case _:
            return "Not a 2D point"

for p in [(0, 0), (0, 5), (3, 0), (2, 7), (1, 2, 3)]:
    print(p, "→", where(p))

(0, 0) → Origin
(0, 5) → On the Y axis at y=5
(3, 0) → On the X axis at x=3
(2, 7) → At (2, 7)
(1, 2, 3) → Not a 2D point


## 5. Mapping patterns (dictionaries)

Great for JSON / API data. Extra keys are ignored. Use `**rest` to capture them.

In [6]:
def process(event):
    match event:
        case {"type": "click", "x": x, "y": y}:
            return f"Click at ({x}, {y})"
        case {"type": "keypress", "key": "Enter" | "Return"}:
            return "Submit!"
        case {"type": "keypress", "key": key}:
            return f"Key {key!r}"
        case {"type": t, **rest}:
            return f"Unhandled {t} with {rest}"
        case _:
            return "Invalid event"

events = [{"type": "click", "x": 10, "y": 20, "button": "left"},
          {"type": "keypress", "key": "Enter"},
          {"type": "keypress", "key": "a"},
          {"type": "scroll", "delta": -3},
          "garbage"]
for e in events:
    print(process(e))

Click at (10, 20)
Submit!
Key 'a'
Unhandled scroll with {'delta': -3}
Invalid event


## 6. Type checks & class patterns

`int()`, `str()`, `list()` etc. in a pattern check the type (and can capture):

In [7]:
def kind(value):
    match value:
        case bool():                 # put bool before int (bool is a subclass of int)
            return "boolean"
        case int() | float():
            return "number"
        case str() as s if s.strip() == "":
            return "blank string"
        case str():
            return "string"
        case [*_]:
            return "sequence"
        case {}:
            return "mapping"
        case None:
            return "nothing"
        case _:
            return "something else"

print([kind(v) for v in [True, 3, 2.5, "  ", "hi", [1], (1, 2), {"a": 1}, None, {1}]])

['boolean', 'number', 'number', 'blank string', 'string', 'sequence', 'sequence', 'mapping', 'nothing', 'something else']


Class patterns match on **attributes** of objects; they pair beautifully with dataclasses (Module 06):

In [8]:
from dataclasses import dataclass

@dataclass
class Circle:
    radius: float

@dataclass
class Rectangle:
    width: float
    height: float

def area(shape):
    match shape:
        case Circle(radius=r):
            return 3.14159 * r ** 2
        case Rectangle(width=w, height=h) if w == h:
            return f"square of area {w * h}"
        case Rectangle(w, h):        # positional works for dataclasses (via __match_args__)
            return w * h

print(area(Circle(1)), area(Rectangle(3, 3)), area(Rectangle(2, 5)))

3.14159 square of area 9 10


## 7. `as`: capture a whole sub-pattern

In [9]:
match [1, [2, 3]]:
    case [first, [_, _] as inner]:
        print(first, inner)

1 [2, 3]


## 8. ❌ The #1 pitfall: matching against a variable

A bare name **always matches and captures**; it doesn't compare against an existing variable!

In [10]:
RED = "red"
color = "blue"
match color:
    case RED:            # ❌ captures 'blue' into RED, so it ALWAYS matches
        print("matched?!", RED)

matched?! blue


To compare against constants, use a **dotted name** (e.g. an `Enum` member or module attribute) or a guard:

In [11]:
from enum import Enum
class Color(Enum):
    RED = "red"
    BLUE = "blue"

def paint(c):
    match c:
        case Color.RED:            # ✅ dotted name → compared by value
            return "Stop"
        case Color.BLUE:
            return "Calm"

print(paint(Color.BLUE))

Calm


## 9. When to use `match` vs `if/elif`
- **`match`**: when you're inspecting the **structure** of data (commands, JSON, ASTs, tuples, objects) and want to destructure it.
- **`if/elif`**: for simple boolean conditions and ranges (`x > 10`).
- For a plain value → value mapping, a **dict** is often simplest.

## 📝 Summary
- `match subject:` + `case pattern:`; first match wins; `case _:` is the default.
- Patterns: literals, `|`, captures, `_`, sequences `[a, *rest]`, mappings `{"k": v}`, classes `Point(x=0)`, `as`, guards `if`.
- Bare names capture! Use dotted names (`Color.RED`) for constants.

## 🏋️ Practice
1. Write `calc(expr)` that takes strings like `"3 + 4"`, `"10 / 2"` and uses `match` on `expr.split()` to compute the answer.
2. Write `traffic(light)` returning an action for `"red"`, `"yellow"`, `"green"`, otherwise `"invalid"`.
3. Given API responses like `{"status": "ok", "data": [...]}` or `{"status": "error", "message": "..."}`, return the data length or the message.

<details><summary>Show solutions</summary>

```python
def calc(expr):
    match expr.split():
        case [a, "+", b]: return float(a) + float(b)
        case [a, "-", b]: return float(a) - float(b)
        case [a, "*", b]: return float(a) * float(b)
        case [a, "/", b] if float(b) != 0: return float(a) / float(b)
        case _: return "invalid"

def traffic(light):
    match light:
        case "red": return "stop"
        case "yellow": return "slow down"
        case "green": return "go"
        case _: return "invalid"

def handle(resp):
    match resp:
        case {"status": "ok", "data": list(items)}: return len(items)
        case {"status": "error", "message": msg}: return msg
        case _: return "unknown"
```
</details>